To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### Unslothが配布したコードの改変

試しに、データセットを使ってGRPOを使った結果をgoogle colabで試せるためのコードです。このコードでは、Qwen3-4B-Base(因果モデル)を、Reasoning能力を獲得するために、SFTによるフォーマット学習を行なっています。
GRPOのためのフォーマット学習(SFT) -> GPROのためのデータセット成形 -> GPROの流れになっています。
T4だと遅いので、モデルを変えるか、load_in_4bit = Trueにするか、別のGPUを使うことをお勧めします。
#### Reasoning能力を獲得したモデルをさらに学習させたほうが性能がいいのかもしれないです。その場合は、Formatを変更するためにSFTを行うか、SFTを飛ばしてGRPO?


In [ ]:
# 2025.7.3のものを使用すること
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth==2025.7.3 vllm==0.8.5.post1

In [ ]:
# 2025.7.3のものを使用すること
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth==2025.7.3 vllm
else:
    !pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo==2025.7.4
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
# エラーが起きた場合Debugしてください
os.environ["TRANSFORMERS_VERBOSITY"] = "debug"
# huggingfaceはかきこみtokenで
# HF_TOKENという名前でkeyマークの環境変数に入れる必要あるかも
# gitはnでいい
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

### Installation

### Unsloth

目標：Thinkが含まれたデータセットを使用して、`Qwen3-4B-Base`をGRPO経由で推論モデルに変換する。
まず、モデルを事前微調整してGRPOがフォーマットの一致を試行しないようにします。これにより、GRPOの処理速度が向上します。

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

# モデルとトークナイザの取得
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

# LoRAアダプタの用意
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

### GRPO チャットテンプレート
ベースモデルを使用しているため、チャットテンプレートを設定する必要があります。独自のチャットテンプレートを作成することも可能です！
1. DeepSeekでは<think>と</think>を使用していますが、これは必須ではありません - 自由にカスタマイズ可能です！
2. system_promptは、モデルの応答をガイドするため、少なくとも設定することをおすすめします



In [ ]:
reasoning_start = "<think>" # Acts as <think>
reasoning_end   = "</think>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

以下のシンプルなチャットテンプレートを作成します。`add_generation_prompt`には、モデルが推論プロセスを開始するようにガイドするため、`<think>`を先頭に追加しています。

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with out specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

例を見て、チャットテンプレートがどのように動作するか確認しましょう：

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

### フォーマット調整のためのSFT
現在、NVIDIAの[Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning)から、高品質なDeepSeek R1トレースのみを含むサブセットを使用しています。
まず、約59件程度の例で、GRPOフォーマットを理解させるための初期調整を行います。

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot", num_proc=1)
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

データセットをGRPOスタイルのフォーマットに準拠するように修正

In [ ]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

Check to see if it worked:

In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

事前微調整前のデータセットを`max_seq_length/2`に切り詰めます。これは、あまりにも長い推論トレースを避けたいからです。
注意：この処理には2分ほどかかる場合があります！

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

次に、メッセージをトークン化し、Hugging Faceと互換性のあるデータセット形式に変換します。補足：これは、フィルタリングのためにpandas DataFrameに変換した形式を再度datasets.Datasetに変換する必要があるからです。：

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

モデルを微調整して、GRPOフォーマットに準拠させる

In [ ]:
from trl import SFTTrainer, SFTConfig
# LoRA部分をSFT
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)


In [ ]:
trainer.train()

モデルがカスタム形式に従って学習しているかどうかを確認：

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

フォーマットにある程度従うことができたので、不要なキャッシュやGCを削除

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

### データ準備
<a name="Data"></a>

ここから、GRPOをおこなうため、データセットのフォーマット化、報酬関数の定義、プロンプト長のフィルタリングを行います。
Hugging Faceの[Open R1 Mathデータセット](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed)を使用しています。また、OpenAIの有名な[GSM8Kデータセット](https://huggingface.co/datasets/openai/gsm8k)も利用可能です。

In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

Let's look at the first row:

In [ ]:
dataset[0]["prompt"]

In [ ]:
dataset[0]["solution"]

In GSM8K, ee notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

Let's map the dataset! and see the first row:

In [ ]:
# データセットを同じフォーマットに修正
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

推論セクションと回答に一致する正規表現形式を作成。正規表現形式をもとに、モデルの作成が期待されるフォーマットに沿っているかを検証する

In [ ]:
import re

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

きちんと形式を取得できるかテスト。答えさえ取得できれば成功

In [ ]:
match_format.findall(
    "Let me think!</think>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

In [ ]:
match_format.findall(
    "<think>Let me think!</think>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

次に、形式を正確に一致させる報酬関数を作成。成功した場合に3ポイントの報酬を付与：

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

失敗した場合、モデルが少なくとも部分的にフォーマットに従っている場合、各シンボルをカウントしてモデルに報酬を与えるようにする：

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # キーワードがいくつあるかを数える(</think>、<SOLUTION>、</SOLUTION>)
        # もし1つ見られたら、0.5点追加 - ２つ以上ある場合はペナルティとして-1.0点を与える
        # <think>は常に前に設置されるため、報酬は与えない。
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

最後に、生成された回答を抽出し、そして、その回答に報酬を付与するか、またはペナルティを課します。また、回答が真の回答にどれだけ近いかを比率で評価し、その評価に基づいて報酬を付与します：
以下では、回答の値でスコアが付与される

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    # 質問の取得
    question = prompts[0][-1]["content"]
    # 回答の取得
    responses = [completion[0]["content"] for completion in completions]

    # 最終回答の部分のみを定義した正規表現をもとに抽出
    # 形式が誤って回答している場合None
    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    # スコア計算と評価
    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        # 形式に沿っていない場合
        if guess is None:
            scores.append(-2.0)
            continue
        # 回答が正解の場合、5点獲得
        if guess == true_answer:
            score += 5.0
        # モデルが抽出した回答の前後の空白を削除した場合、5点に-1.5点のペナルティ
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # 回答が比率によって近い場合にも報酬を与える
            # つまり、回答が何らかの範囲内にある場合、報酬を与えます！
            try:
                ratio = float(guess) / float(true_answer)
                # +-10%以下の比率であればスコアを2点追加
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                # +-20%以下の比率であればスコアを1.5点追加
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # 大きく間違った場合
            # 上記のtryブロック内でエラー（ゼロ除算、非数値の変換試行など）が発生した場合
            except:
                score -= 4.5 # ペナルティ
        scores.append(score)
    return scores

また、答えが1つの数字ではなく、例えば「解決策は$20です」のような文の場合もあります。その場合、20を抽出します。


In [ ]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34点  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456億円  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234$  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

デバックのために、生成された応答と正しい回答を出力するメイン関数を作成。また、テキストを`float`型に変換し、それが同じかどうかを確認する別の報酬関数も作成。これは、数値回答の堅牢な評価、カンマなどの書式対応、そしてモデルが数値的な回答を生成しない場合の厳格なペナルティを適用するため

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

# スコア計算
def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    # 推論から取得したanswerとtrue_answerを比較
    for guess, true_answer in zip(extracted_responses, answer):
        # パターン通りに回答できなかった場合-2.5点
        if guess is None:
            scores.append(-2.5)
            continue
        # 数値についた余分なものを排除(,や など)
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            # 余分なものを排除した結果正解ならば3.5点追加
            # 不正解なら-1.5(形式はあっていた)
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            # 回答が数値でなかったり解析ができなかった場合0
            scores.append(0)
            continue
    return scores

上位10%の長いプロンプトを削除して、無駄な長文を生成させることを抑える

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

<a name="GRPOの学習"></a>
### Train the model

GRPO Trainer を設定

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
# GRPOを用いてLoraアダプタを強化学習(SFT後のLoRAアダプタ)
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

トレーナーを実行しましょう！画面を上へスクロールすると、報酬の表が表示されます。目標は「報酬」列の数値が増えることです
アクションが発生するまで150～200ステップ待つ必要があります。最初の100ステップでは報酬が0になる可能性があります。
報酬が上がっていなければ、どこかミスっている、実行し忘れている可能性が高いです。

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)
# 実行できる時とできない時があるので注意。もう一度実行すれば、なんとかなる
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

<a name="推論"></a>
### Inference
GRPOを訓練したモデルとGRPOを実行していないモデル(SFTのみ)を比較してみる。まず、GRPOを実行していないモデルを試してみる

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

GRPOでトレーニングしたLoRAを保存

In [ ]:
model.save_lora("grpo_saved_lora")

LoRAモデルの重みファイルが実際に学習されており、完全にゼロではないことを検証

In [ ]:
# 検証
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

 GRPO済みのモデルで推論

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

推論モデルははるかに優れている。
 シーケンスの長さを延長し、より長い時間訓練すれば、さらに改善される

<a name="Save"></a>
### VLLM用にfloat16形式で保存
float16形式への直接保存もサポート。float16の場合は`merged_16bit`、int4の場合は`merged_4bit`を選択。また、`lora`アダプターをフォールバックとして使用可能です。Hugging Faceアカウントにアップロードするには`push_to_hub_merged`を使用する必要がある。個人用トークンは https://huggingface.co/settings/tokens から取得できます。

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
